# Notebook C — Cellular Heterogeneity [OPTIONAL]

**Notebook C — High School Internship · Quantitative Microbiology Lab**

This notebook is the optional follow-up to **Notebook B** (Gene Expression and Regulation).
It explores two related questions:
1. **§C1 — Bistability**: why does TMG produce a bimodal fluorescence distribution?
2. **§C2 — ImageJ analysis**: how do we quantify this heterogeneity from microscopy images?

---

> **Prerequisite**: completed §B3 of Notebook B.
> **Environment**: [Google Colab](https://colab.research.google.com)
> **Estimated duration**: 1–2 half-days


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size']  = 12
print("Libraries loaded.")

## Section C1 — LacY / TMG Bistability

TMG is transported into the cell by the **LacY** protein — itself encoded by
the *lac* operon. This creates a **positive feedback loop**:

```
More LacY → more intracellular TMG → more active promoter → more LacY
```

This loop can create **two stable states**:
- **OFF**: little LacY → little intracellular TMG → weakly active promoter
- **ON**: much LacY → much TMG → highly active promoter

At **intermediate** TMG concentrations, both states coexist.
Two identical cells in the same medium can end up in different states
depending on their initial state — this is **bistability**.

> **Link with §A**: in §A, stochastic noise created variability.
> Here, bistability creates a bimodal distribution: even without noise,
> two identical cells can diverge.


In [ ]:
# ── Parameters ─────────────────────────────────────────────────────────────────
K_bis   = 50      # demi-saturation (µM TMG intracellulaire)
gain    = 8       # amplification du TMG par LacY (retro-action)
alpha   = 0.05    # vitesse de relaxation de LacY
n_t     = 400     # nombre de pas de temps
tmg_ext = 20      # µM TMG extracellulaire — zone bistable

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── Left: two identical cells, different initial states ─────────────────────────
for lacy_init, couleur, etiquette in [
        (0.05, 'steelblue', 'Cell A  (starting low)'),
        (0.85, 'crimson',   'Cell B  (starting high)')]:
    lacy = lacy_init
    traj = [lacy]
    for t in range(n_t):
        tmg_eff  = tmg_ext * (1 + gain * lacy)      # LacY amplifies intracellular TMG
        p_on_bis = tmg_eff / (K_bis + tmg_eff)      # promoter response
        lacy     = lacy + alpha * (p_on_bis - lacy)  # progressive relaxation
        lacy     = max(0.0, min(1.0, lacy))
        traj.append(lacy)
    axes[0].plot(traj, color=couleur, lw=2, label=etiquette)

axes[0].axhline(0.5, color='gray', linestyle='--', alpha=0.5)
axes[0].set_xlabel("Time step")
axes[0].set_ylabel("LacY level (relative expression)")
axes[0].set_title(f"Two identical cells, [TMG] = {tmg_ext} µM")
axes[0].legend()

# ── Right: distribution in a population ─────────────────────────────────────────
N_pop = 200
lacy_finales = []
for _ in range(N_pop):
    lacy = np.random.uniform(0, 1)    # random initial state
    for t in range(n_t):
        tmg_eff  = tmg_ext * (1 + gain * lacy)
        p_on_bis = tmg_eff / (K_bis + tmg_eff)
        lacy     = lacy + alpha * (p_on_bis - lacy)
        lacy     = max(0.0, min(1.0, lacy))
    lacy_finales.append(lacy)

axes[1].hist(lacy_finales, bins=np.linspace(0, 1, 26),
             color='mediumpurple', edgecolor='white', alpha=0.85)
axes[1].set_xlabel("Final LacY level")
axes[1].set_ylabel("Number of cells")
axes[1].set_title(f"Population of {N_pop} cells, [TMG] = {tmg_ext} µM")

plt.tight_layout()
plt.show()

print("-> Bimodal distribution: each cell converges to one of the two stable states.")
print()
print("   Try tmg_ext = 5, 20, 50, 100 µM.")
print("   At which concentration does the distribution become unimodal?")

## Section C2 — Microscopy data analysis (ImageJ)

Microscopy allows us to measure the fluorescence of **individual cells** and
directly observe the distribution in the population.

**Export from ImageJ**:
1. Open the image, segment cells (*Analyze Particles*)
2. *Measure* → export the table as CSV (one row per cell)
3. Repeat for each TMG concentration

The useful column in the ImageJ CSV is `Mean` (mean fluorescence per cell).


In [ ]:
# ── Load your ImageJ data ───────────────────────────────────────────────────────
# concs_tmg = [0, 5, 20, 100]   # µM TMG
# donnees_ij = {conc: pd.read_csv(f'imagej_tmg_{conc}uM.csv') for conc in concs_tmg}
# For each condition: donnees_ij[conc]['Mean'] = per-cell fluorescence

# ── Synthetic data (replace with your real ImageJ data) ─────────────────────────
np.random.seed(7)

def simuler_imagej(tmg_ext, n_cells=150):
    # Simulates single-cell fluorescence distribution for a given [TMG].
    K_s, gain_s, alpha_s, n_t_s = 50, 8, 0.05, 400
    fl = []
    for _ in range(n_cells):
        lacy = np.random.uniform(0, 1)
        for t in range(n_t_s):
            tmg_eff = tmg_ext * (1 + gain_s * lacy)
            p       = tmg_eff / (K_s + tmg_eff)
            lacy    = lacy + alpha_s * (p - lacy)
            lacy    = max(0.0, min(1.0, lacy))
        fl.append(lacy * 1000 + np.random.normal(0, 30))
    return np.array(fl)

concs_tmg   = [0, 5, 20, 100]
donnees_sim = {c: simuler_imagej(c) for c in concs_tmg}

# ── Histograms by concentration ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
couleurs_d = ['steelblue', 'mediumseagreen', 'darkorange', 'crimson']

for i, conc in enumerate(concs_tmg):
    fl = donnees_sim[conc]
    # For real data: fl = donnees_ij[conc]['Mean']

    ax = axes[i // 2][i % 2]
    ax.hist(fl, bins=30, color=couleurs_d[i], edgecolor='white', alpha=0.85)
    ax.set_xlabel("Fluorescence (a.u.)")
    ax.set_ylabel("Number of cells")
    ax.set_title(f"[TMG] = {conc} µM  (n = {len(fl)} cells)")

plt.suptitle("Single-cell fluorescence distribution (TMG, microscopy)", fontsize=13)
plt.tight_layout()
plt.show()

print("Questions:")
print("  1. At which [TMG] do you see two separate peaks?")
print("  2. Does the fraction of ON cells increase with [TMG]?")
print("  3. Do these results match the §C1 model predictions?")

---
## Going further

1. **Bimodal distribution and noise**: how do we experimentally distinguish
   bistability (two stable states) from high stochastic noise?
   *Hint*: redo the experiment with two different inducers (IPTG vs TMG)
   and compare the distributions.

2. **Connection with Elowitz 2002**: the single-cell analysis you performed
   is close to the foundational noise experiment.
   In that paper, two identical reporter genes in the same cell showed
   different expression levels — direct proof of *intrinsic* noise.

3. **Quantify bimodality**: can you measure the fraction of ON cells
   for each TMG concentration and plot this fraction as a function of [TMG]?
   What should the bistability model from §C1 predict?
